# 07 — Mejoras: comparar antes de adoptar

Este último notebook reúne las extensiones del ejercicio. La línea de base ya mostró errores de recuperación, autoridad, abstención y formato. En vez de modificar todo simultáneamente, organizamos experimentos separados.

| Experimento | Qué cambia | Qué se mantiene |
|---|---|---|
| Recuperación | BM25 o fusión RRF frente a dense | Mismos textos, preguntas y k |
| Metadatos | Filtro explícito de documentos normativos actuales | Mismo ranking antes de filtrar |
| Formato | JSON Schema generado con Pydantic | Modelo, prompt original y contexto exacto de la corrida base |
| Instrucciones | Reglas más precisas de alcance y abstención | Mismo esquema, modelo y contexto que la variante estructurada |

**Por defecto `EJECUTAR_API=True` genera tus resultados y hace veinte llamadas:** diez con esquema y prompt original, y diez con esquema e instrucciones revisadas. La búsqueda y los filtros se evalúan localmente y no se mezclan con esas llamadas. Esto permite atribuir diferencias con mayor claridad.

No prometemos que híbrido gane ni que Pydantic haga verdaderas las respuestas. Los cambios están motivados por casos que ya vimos: esta es una evaluación de desarrollo, no una medición independiente de generalización. No ajustamos parámetros repetidamente hasta que las diez preguntas pasen.

Ejecutar en orden con Python (rag-demo). Si faltan archivos del índice, ejecutar los pasos anteriores. Los notebooks 01–06 y la rúbrica permanecen como referencia.

## 1. Cargar la misma preparación

La matriz y los chunks son los ya utilizados. Repetimos validaciones de huellas, orden, forma y normalización: cualquier comparación carece de sentido si se usan fuentes distintas sin advertirlo. Conservamos también el recuperador NumPy como referencia.

Las funciones quedan visibles en este notebook; no ejecutamos otros notebooks ni sus celdas de API. La carga del modelo de embeddings se realiza desde la caché local.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import numpy as np
from IPython.display import display, Markdown

actual = Path.cwd().resolve()
raiz = next((p for p in [actual, *actual.parents]
             if (p / "requirements.in").is_file() and (p / "corpus").is_dir()), None)
if raiz is None:
    raise RuntimeError("Abrí el notebook dentro de rag-demo.")
ruta_chunks = raiz / "data/chunks.jsonl"
chunks = [json.loads(linea) for linea in ruta_chunks.read_text(encoding="utf-8").splitlines()]
manifiesto = json.loads((raiz / "data/manifest.json").read_text(encoding="utf-8"))
if not chunks or len(chunks) != manifiesto["chunks"]:
    raise ValueError("Exportación incompleta: ejecutar notebook 02.")
if len({c["chunk_id"] for c in chunks}) != len(chunks):
    raise ValueError("IDs de chunk duplicados.")
fuentes = {s["source"]: s["sha256"] for s in manifiesto["sources"]}
actuales = {p.relative_to(raiz).as_posix() for p in (raiz / "corpus").glob("*.md")}
if actuales != set(fuentes):
    raise ValueError("Cambió el conjunto de fuentes; ejecutar notebook 02.")
for fuente, huella in fuentes.items():
    if hashlib.sha256((raiz / fuente).read_bytes()).hexdigest() != huella:
        raise ValueError(f"Cambió {fuente}; ejecutar notebook 02.")
for c in chunks:
    if c["source_sha256"] != fuentes[c["source"]]:
        raise ValueError("El chunk y el manifiesto no corresponden a la misma fuente.")
print("Chunks cargados:", len(chunks))
print("Primer ID:", chunks[0]["chunk_id"])
config_indice = json.loads((raiz / "data/embeddings_manifest.json").read_text(encoding="utf-8"))
embeddings = np.load(raiz / "data/embeddings.npy", allow_pickle=False)
if config_indice["chunks_sha256"] != hashlib.sha256(ruta_chunks.read_bytes()).hexdigest():
    raise ValueError("Los embeddings corresponden a otros chunks: ejecutar notebook 03.")
if config_indice["chunk_ids"] != [c["chunk_id"] for c in chunks]:
    raise ValueError("El orden de las filas no coincide con los chunks.")
if list(embeddings.shape) != config_indice["shape"] or embeddings.shape[0] != len(chunks):
    raise ValueError("La forma de la matriz no coincide con el manifiesto.")
if not config_indice["normalized"] or not np.isfinite(embeddings).all():
    raise ValueError("Matriz no normalizada o con valores inválidos.")
if not np.allclose(np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5):
    raise ValueError("Las filas deben tener norma 1.")
print("Matriz lista:", embeddings.shape)

os.environ["HF_HOME"] = str(raiz / ".cache/huggingface")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
from sentence_transformers import SentenceTransformer
import torch

torch.set_num_threads(min(4, os.cpu_count() or 1))
modelo_embeddings = SentenceTransformer(
    config_indice["model"], revision=config_indice["revision"], device="cpu",
    cache_folder=str(raiz / ".cache/huggingface/hub"),
    local_files_only=True, trust_remote_code=False, token=False,
)
if modelo_embeddings.get_embedding_dimension() != embeddings.shape[1]:
    raise ValueError("Dimensiones incompatibles entre modelo e índice.")

def recuperar(pregunta, k=3):
    if not isinstance(pregunta, str) or not pregunta.strip():
        raise ValueError("La pregunta no puede estar vacía.")
    if isinstance(k, bool) or not isinstance(k, int) or k < 1:
        raise ValueError("k debe ser un entero positivo.")
    texto = config_indice["query_prefix"] + pregunta.strip()
    if len(modelo_embeddings.tokenizer(texto, truncation=False)["input_ids"]) > modelo_embeddings.max_seq_length:
        raise ValueError("Consulta demasiado larga para el modelo de embeddings.")
    vector = modelo_embeddings.encode(texto, normalize_embeddings=True, convert_to_numpy=True)
    scores = embeddings @ vector
    indices = np.argsort(-scores, kind="stable")[:min(k, len(chunks))]
    return [{"rank": rango, "score": float(scores[i]), "chunk": chunks[int(i)]}
            for rango, i in enumerate(indices, start=1)]

print("Recuperador local listo:", config_indice["model"])

**Qué esperar:** 17 fragmentos, matriz 17 × 384 y modelo E5 listo. Esta celda no envía consultas a OpenAI.

## 2. Fijar la corrida de referencia

Leemos `eval/results/latest.json` para seleccionar tu última evaluación del notebook 05. Su identificador se conserva en `BASE_RUN` durante esta ejecución. Si generas otra evaluación base, vuelve a ejecutar este notebook desde el principio; una comparación guardada con otra base será rechazada por su firma. Verificamos corpus, matriz, preguntas y rúbrica contra sus huellas.

Para generación reutilizaremos literalmente `input` de cada caso base. Esto mantiene iguales los fragmentos, orden y texto, incluso si un nuevo recuperador ofrece un ranking mejor. No copiamos las respuestas esperadas al prompt: solo se consultan después para medir.

La bandera de API está en True para generar tus veinte respuestas. Después puedes usar False para revisar tu propia ejecución sin nuevas llamadas. `K=3` y la constante de RRF se fijan antes de mirar los resultados; no se optimizan contra la rúbrica.

In [ ]:
import yaml
from datetime import date, datetime, timezone
from uuid import uuid4

# La evaluación creada por el alumno en el notebook 05 es la referencia.
ruta_base = raiz / "eval/results/latest.json"
if not ruta_base.exists():
    raise FileNotFoundError("Primero ejecutar el notebook 05 con EJECUTAR_API=True.")
BASE_RUN = json.loads(ruta_base.read_text(encoding="utf-8"))["run"]
base = raiz / "eval/results" / BASE_RUN
config_base = json.loads((base / "config.json").read_text(encoding="utf-8"))
registros_base = {f"Q{i:02}": json.loads((base / f"Q{i:02}.json").read_text(encoding="utf-8")) for i in range(1, 11)}
for nombre, ruta in {
    "chunks": ruta_chunks, "embeddings": raiz / "data/embeddings.npy",
    "index_manifest": raiz / "data/embeddings_manifest.json",
    "questions": raiz / "eval/preguntas.md", "rubric": raiz / "eval/expected_results.yaml",
}.items():
    if hashlib.sha256(ruta.read_bytes()).hexdigest() != config_base["signature"][nombre]:
        raise ValueError(f"Cambió {nombre}; definir una nueva línea de base antes de comparar.")
rubrica = yaml.safe_load((raiz / "eval/expected_results.yaml").read_text(encoding="utf-8"))
casos = {r["id"]: r for r in rubrica["cases"]}
preguntas = {id: r["question"] for id, r in registros_base.items()}
if set(casos) != set(preguntas):
    raise ValueError("No coinciden los casos de referencia y rúbrica.")
K = 3
RRF_CONSTANTE = 60
EJECUTAR_API = True
INSTRUCCIONES_BASE = config_base["signature"]["instructions"]
MODELO_LLM = config_base["llm_requested"]
print("Referencia:", BASE_RUN, "Modelo:", MODELO_LLM)

## 3. Añadir BM25 sobre los mismos textos

BM25 puntúa coincidencias léxicas teniendo en cuenta frecuencia y longitud. No requiere embeddings ni conoce la semántica de una paráfrasis. Puede funcionar bien cuando consulta y documento comparten términos precisos.

Creamos un almacén distinto del usado en la demo de Haystack: su `content` contiene exactamente el texto compacto que se codificó con E5, sin el prefijo `passage: `. No indexamos las respuestas esperadas. Así la comparación de recuperación cambia el método, no los textos disponibles.

Elegimos BM25Okapi y un patrón de palabras Unicode; no introducimos stemming, traducción ni expansión de consulta. La puntuación BM25 **no está en la misma escala que el coseno**. Sumarlas directamente sería una decisión arbitraria: en el siguiente bloque combinaremos posiciones.

In [ ]:
os.environ["HAYSTACK_TELEMETRY_ENABLED"] = "False"
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

prefijo = config_indice["document_prefix"]
textos_indice = config_indice["embedding_texts"]
if len(textos_indice) != len(chunks) or not all(t.startswith(prefijo) for t in textos_indice):
    raise ValueError("Los textos del índice no corresponden al formato esperado.")
almacen_lexico = InMemoryDocumentStore(
    bm25_algorithm="BM25Okapi", bm25_tokenization_regex=r"(?u)\b\w+\b", shared=False,
)
almacen_lexico.write_documents([
    Document(id=c["chunk_id"], content=t[len(prefijo):]) for c, t in zip(chunks, textos_indice)
])
bm25 = InMemoryBM25Retriever(document_store=almacen_lexico, top_k=len(chunks), scale_score=False)
por_id = {c["chunk_id"]: c for c in chunks}

def ranking_bm25(pregunta):
    documentos = bm25.run(query=pregunta)["documents"]
    # Descendemos por score y usamos el ID como desempate explícito.
    documentos = sorted(documentos, key=lambda d: (-d.score, d.id))
    return [{"chunk_id": d.id, "score": float(d.score)} for d in documentos]

print("Documentos léxicos:", almacen_lexico.count_documents())
print("Q01 BM25:", ranking_bm25(preguntas["Q01"])[:K])

## 4. Combinar rankings con RRF

Reciprocal Rank Fusion asigna a un documento la suma de `1 / (C + posición)` en cada lista en que aparece. Las posiciones comienzan en 1. Con C=60, las diferencias entre posiciones cercanas pesan moderadamente. Es una constante de este experimento, no un umbral de verdad ni un parámetro optimizado.

RRF no mira los scores originales; combina ordenamientos. Un fragmento bien ubicado en ambas listas acumula más. Si falta en una lista, esa lista no le aporta puntos. `dict.get(id, 0.0)` inicializa el acumulado y `sorted` ordena por score fusionado, con ID para desempates.

En este corpus pequeño usamos hasta los 17 candidatos por método. En colecciones grandes habría que elegir una profundidad de candidatos. La comparación mantiene top-3 final y cuenta documentos únicos, como el notebook 05. Q08 sigue sin cobertura aplicable.

In [ ]:
def fusion_rrf(*rankings, constante=60):
    if constante < 0:
        raise ValueError("La constante de RRF debe ser no negativa.")
    acumulados = {}
    for ranking in rankings:
        vistos = set()
        for posicion, r in enumerate(ranking, start=1):
            id = r["chunk_id"]
            if id in vistos:
                raise ValueError("Un ranking contiene IDs duplicados.")
            vistos.add(id)
            acumulados[id] = acumulados.get(id, 0.0) + 1.0 / (constante + posicion)
    return [{"chunk_id": id, "score": score} for id, score in
            sorted(acumulados.items(), key=lambda par: (-par[1], par[0]))]

def cobertura(ranking, id, k=3):
    esperados = set(casos[id]["relevant_documents"])
    obtenidos = {por_id[r["chunk_id"]]["document_id"] for r in ranking[:k]}
    return len(obtenidos & esperados) / len(esperados) if esperados else None

rankings = {}
comparacion_retrieval = []
for id, pregunta in preguntas.items():
    dense = [{"chunk_id": r["chunk"]["chunk_id"], "score": r["score"]}
             for r in recuperar(pregunta, len(chunks))]
    lexico = ranking_bm25(pregunta)
    rankings[id] = {"dense": dense, "bm25": lexico,
                    "rrf": fusion_rrf(dense, lexico, constante=RRF_CONSTANTE)}
    for metodo, ranking in rankings[id].items():
        valor = cobertura(ranking, id, K)
        comparacion_retrieval.append({"case_id": id, "method": metodo, "k": K,
                                     "document_coverage": valor, "top_ids": [r["chunk_id"] for r in ranking[:K]]})
        print(id, metodo, "cobertura:", valor, "top:", [r["chunk_id"] for r in ranking[:K]])
for metodo in ["dense", "bm25", "rrf"]:
    valores = [r["document_coverage"] for r in comparacion_retrieval if r["method"] == metodo and r["document_coverage"] is not None]
    print(metodo, "media documental:", round(float(np.mean(valores)), 3), "casos:", len(valores))

**Qué mirar:** comparar Q01 y Q02 antes de mirar solo el promedio. Una mejora media puede ocultar una pérdida en un caso importante. La cobertura documental no exige la sección exacta; inspeccionar los IDs. No se declara un ganador semántico ni se adopta híbrido automáticamente.

## 5. Usar metadatos sin borrar la historia por accidente

Un filtro de documentos normativos actuales puede quitar FAQ y versiones expiradas, pero Q03 necesita explicar una regla histórica. Por eso el filtro es una opción explícita, no una detección oculta basada en palabras ni en el ID de evaluación.

La función usa fecha, estado y autoridad. Mantiene las políticas globales y anexos actuales; no infiere país de contratación a partir del destino. Que un anexo sobreviva al filtro no significa que aplique al usuario: esa condición sigue requiriendo evidencia.

Probamos dos intenciones: `todas` y `normativa_actual`. Son una demostración del costo del filtro, no un router automático. No convertimos `vigente` en una regla de superioridad universal: una FAQ publicada puede estar vigente y tener menor autoridad. La fecha de consulta queda fija para reproducibilidad.

In [ ]:
FECHA_CONSULTA = date.fromisoformat(rubrica["as_of_date"])

def aplicar_filtro(ranking, modo="todas"):
    if modo not in {"todas", "normativa_actual"}:
        raise ValueError("Modo desconocido.")
    if modo == "todas":
        return list(ranking)
    filtrados = []
    for r in ranking:
        m = por_id[r["chunk_id"]]["metadata"]
        fecha_ok = date.fromisoformat(m["vigente_desde"]) <= FECHA_CONSULTA
        fecha_ok = fecha_ok and (m["vigente_hasta"] is None or FECHA_CONSULTA <= date.fromisoformat(m["vigente_hasta"]))
        if fecha_ok and m["estado"] == "vigente" and m["autoridad"] in {"politica_normativa", "anexo_normativo"}:
            filtrados.append(r)
    return filtrados

experimentos_filtro = []
for id in ["Q01", "Q03"]:
    for modo in ["todas", "normativa_actual"]:
        ranking = aplicar_filtro(rankings[id]["rrf"], modo)
        resultado = {"case_id": id, "mode": modo, "top_ids": [r["chunk_id"] for r in ranking[:K]],
                     "document_coverage": cobertura(ranking, id, K)}
        experimentos_filtro.append(resultado)
        print(resultado)

**Interpretación:** excluir V2 puede facilitar una respuesta actual y a la vez impedir explicar de dónde salió la regla de 30 días. No alcanza con «filtrar lo viejo» para resolver todo RAG. Esta variante no se aplicará a las llamadas de generación que siguen: conservarán exactamente el contexto original.

## 6. Declarar la estructura con Pydantic

`BaseModel` describe campos y tipos. `Literal` restringe los estados permitidos; `list[str]` pide listas de textos. `extra="forbid"` impide campos inesperados y `strict=True` evita conversiones de tipos al validar localmente. Ningún campo tiene valor predeterminado: todos deben estar presentes.

`model_json_schema()` produce un JSON Schema que enviamos a Responses API con `strict=True`. Eso condiciona la forma de una respuesta normal completada; negativas del servicio y salidas incompletas deben manejarse aparte. No garantiza hechos correctos, selección de fuentes ni coherencia lógica entre campos.

Mantenemos el chequeo local anterior para citas y contradicciones entre estado e información faltante. Separar ambos niveles permite observar qué arregla el esquema y qué no. No se codifican los montos o plazos correctos como validaciones, porque eso introduciría la respuesta del examen en el sistema.

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, ValidationError

class RespuestaRAG(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    status: Literal["answered", "insufficient_evidence", "no_answer"]
    answer: str
    cited_chunk_ids: list[str]
    missing_information: list[str]

ESQUEMA = RespuestaRAG.model_json_schema()
print(json.dumps(ESQUEMA, ensure_ascii=False, indent=2))
def validar_salida(texto, ids_permitidos):
    try:
        dato = json.loads(texto)
    except json.JSONDecodeError:
        return None, ["La salida no es JSON válido."]
    campos = {"status", "answer", "cited_chunk_ids", "missing_information"}
    if not isinstance(dato, dict) or set(dato) != campos:
        return None, ["El objeto no tiene exactamente los cuatro campos esperados."]
    if not isinstance(dato["status"], str) or dato["status"] not in {"answered", "no_answer", "insufficient_evidence"}:
        return None, ["Estado inválido."]
    if not isinstance(dato["answer"], str) or not dato["answer"].strip():
        return None, ["answer debe ser texto no vacío."]
    for campo in ["cited_chunk_ids", "missing_information"]:
        if not isinstance(dato[campo], list) or not all(isinstance(x, str) for x in dato[campo]):
            return None, [f"{campo} debe ser una lista de textos."]
    citadas = set(dato["cited_chunk_ids"])
    inline = set(re.findall(r"\[([^\[\]]+)\]", dato["answer"]))
    problemas = []
    if citadas - ids_permitidos:
        problemas.append("Hay citas a IDs que no se enviaron.")
    if inline != citadas:
        problemas.append("Las citas inline no coinciden con cited_chunk_ids.")
    if dato["status"] == "answered" and not citadas:
        problemas.append("Una respuesta normativa debe citar evidencia.")
    if dato["status"] == "insufficient_evidence" and not dato["missing_information"]:
        problemas.append("Falta indicar qué información se necesita.")
    if dato["status"] == "answered" and dato["missing_information"]:
        problemas.append("Estado answered incompatible con información declarada como faltante.")
    return dato, problemas


## 7. Modificar instrucciones sin escribir las respuestas esperadas

Definimos dos variantes. `schema_only` mantiene el prompt original y agrega solo el contrato estructurado. `schema_prompt` conserva ese contrato y añade reglas generales sobre evidencia ausente, alcance documental y solicitudes de aclaración.

La variante revisada no contiene «60 días», montos ni IDs de casos. Sin embargo, está motivada por fallos que ya vimos; por eso no es una prueba independiente. Tampoco puede recuperar información que no se envió.

La comparación base → esquema usa una corrida previa, no una réplica simultánea; las salidas pueden variar. Esquema → esquema+prompt mantiene contexto y modelo iguales, pero sigue siendo una muestra pequeña de una ejecución por caso. No atribuir toda diferencia con certeza causal a una instrucción.

In [ ]:
REGLAS_ADICIONALES = """
Antes de declarar answered, comprobá que podés contestar el caso individual con los datos dados.
Si falta una fecha, categoría de gasto, medio de pago u otro dato necesario para elegir o aplicar
una regla, usá insufficient_evidence, hacé la pregunta aclaratoria en answer y enumerá el dato
faltante en missing_information. Una explicación condicional no reemplaza esa aclaración.
Si el contexto no define el beneficio o porcentaje consultado, usá no_answer. Una frase como
'esta política no establece X' delimita el alcance de ese documento: no prueba 'la empresa
no ofrece X', ni permite inferir cero. No transformes ausencia de regulación en prohibición.
Una FAQ de orientación no basta para afirmar una regla normativa actual cuando falta la
política aplicable. Indicá qué fuente falta en vez de presentar la FAQ como autoridad definitiva.
No inventes diferencias entre trámites que las fuentes no establecen. Conservá condiciones,
unidades, alcance, inclusividad de límites y requisitos relevantes de la regla usada.
Si el estado es answered, missing_information debe estar vacío. Si no se puede determinar
un hecho, no lo completes usando conocimiento general. Cada cita inline debe tener un solo
ID entre corchetes; para dos fuentes usá dos pares separados de corchetes.
"""
VARIANTES = {"schema_only": INSTRUCCIONES_BASE,
             "schema_prompt": INSTRUCCIONES_BASE + "\n" + REGLAS_ADICIONALES}
print("Variantes:", list(VARIANTES))
print("Modelo fijo:", MODELO_LLM, "Evidencia: input exacto de la corrida base")

## 8. Ejecutar veinte llamadas o cargar resultados guardados

Con `EJECUTAR_API=False` cargamos un experimento guardado y verificamos su firma. Con True creamos una carpeta nueva, ejecutamos diez casos por variante y guardamos cada respuesta inmediatamente. La línea de base no se sobrescribe.

La función pide el esquema, luego comprueba que la respuesta esté completa y tenga texto JSON. Valida con Pydantic y con el chequeo de citas del paso 6. Un problema técnico o negativa no se convierte en no_answer. Los errores se conservan y no se reintenta hasta obtener un resultado favorable.

Se usa el modelo de la corrida base, aunque `.env` tenga otra preferencia, para mantener la comparación. La clave solo se usa para autenticar. Las respuestas y sus entradas se guardan sin credenciales. Al finalizar, `latest.json` apunta al experimento completo; si se interrumpe a mitad, los archivos parciales permanecen sin reemplazar esa referencia.

In [ ]:
from openai import OpenAI, APIError, APIStatusError
from dotenv import dotenv_values

firma = {"base_run": BASE_RUN, "base_config_sha256": hashlib.sha256((base / "config.json").read_bytes()).hexdigest(),
         "base_records_sha256": {id: hashlib.sha256((base / f"{id}.json").read_bytes()).hexdigest() for id in casos},
         "model": MODELO_LLM, "schema": ESQUEMA, "instructions": VARIANTES, "max_output_tokens": 700}
carpeta_mejoras = raiz / "eval/improvements"
carpeta_mejoras.mkdir(exist_ok=True)

def consultar_estructurado(cliente, entrada, instrucciones, permitidos):
    try:
        r = cliente.responses.create(model=MODELO_LLM, instructions=instrucciones, input=entrada,
            max_output_tokens=700, store=False,
            text={"format": {"type": "json_schema", "name": "respuesta_rag", "schema": ESQUEMA, "strict": True}})
    except APIStatusError as error:
        return {"error": f"HTTP {error.status_code}", "parsed": None, "schema_ok": False, "issues": [], "usage": None}
    except APIError:
        return {"error": "Error de conexión/API", "parsed": None, "schema_ok": False, "issues": [], "usage": None}
    salida = {"error": None, "response_text": r.output_text, "parsed": None, "schema_ok": False,
              "issues": [], "usage": r.usage.model_dump() if r.usage else None, "model": r.model}
    if r.status != "completed" or not r.output_text:
        salida["error"] = "Salida incompleta o sin texto JSON (posible negativa)."
        return salida
    try:
        dato = RespuestaRAG.model_validate_json(r.output_text).model_dump()
    except ValidationError:
        salida["issues"] = ["No cumple el esquema Pydantic."]
        return salida
    _, problemas = validar_salida(r.output_text, permitidos)
    salida.update(parsed=dato, schema_ok=True, issues=problemas)
    return salida

if EJECUTAR_API:
    config = dotenv_values(raiz / ".env", encoding="utf-8-sig")
    if not (config.get("OPENAI_API_KEY") or "").strip():
        raise RuntimeError("Falta OPENAI_API_KEY en .env.")
    corrida = carpeta_mejoras / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])
    corrida.mkdir()
    (corrida / "config.json").write_text(json.dumps(firma, ensure_ascii=False, indent=2), encoding="utf-8")
    resultados = []
    with OpenAI(api_key=config["OPENAI_API_KEY"], base_url="https://api.openai.com/v1", timeout=45.0, max_retries=0) as cliente:
        for variante, instrucciones in VARIANTES.items():
            for id, registro_base in registros_base.items():
                permitidos = {r["chunk_id"] for r in registro_base["retrieved"]}
                registro = consultar_estructurado(cliente, registro_base["input"], instrucciones, permitidos)
                registro.update(case_id=id, variant=variante, input=registro_base["input"], sent_chunk_ids=sorted(permitidos))
                (corrida / f"{variante}_{id}.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
                resultados.append(registro)
                print(variante, id, "error:", registro["error"], "problemas:", registro["issues"])
    (carpeta_mejoras / "latest.json").write_text(json.dumps({"run": corrida.name}), encoding="utf-8")
else:
    if not (carpeta_mejoras / "latest.json").exists():
        raise FileNotFoundError("Aún no hay mejoras guardadas. Usar EJECUTAR_API=True la primera vez.")
    corrida = carpeta_mejoras / json.loads((carpeta_mejoras / "latest.json").read_text(encoding="utf-8"))["run"]
    guardada = json.loads((corrida / "config.json").read_text(encoding="utf-8"))
    if guardada != firma:
        raise ValueError("Los resultados guardados pertenecen a otra configuración.")
    resultados = [json.loads((corrida / f"{v}_{id}.json").read_text(encoding="utf-8")) for v in VARIANTES for id in casos]
    print("Resultados cargados sin API:", corrida.name)

## 9. Comparar métricas sin confundirlas

Contamos cuatro cosas distintas: fallos técnicos, cumplimiento del esquema, validez local de citas/coherencia y coincidencia de estados. Todos usan denominador diez por variante, incluyendo fallos. El esquema no verifica el contenido de `answer`; una afirmación incorrecta puede ajustarse perfectamente a él.

También aplicamos el mismo validador a las respuestas originales para que la columna de consistencia use la misma definición. El estado correcto no prueba que estén todos los hechos. Una abstención en Q02 puede ser más prudente con el contexto disponible, pero sigue siendo un fallo extremo a extremo si el corpus sí contenía la respuesta.

Para juzgar contenido se necesita revisar respuesta y fuentes. Guardamos fichas por criterio de la rúbrica para ambas variantes y no reemplazamos pendientes con aprobados.

In [ ]:
resumen = []
for variante in ["baseline", *VARIANTES]:
    schema_ok = contract_ok = status_ok = errores = 0
    for id in casos:
        if variante == "baseline":
            b = registros_base[id]
            r = {"error": b["api_error"], "parsed": b["parsed"], "response_text": b["response_text"]}
        else:
            r = next(r for r in resultados if r["variant"] == variante and r["case_id"] == id)
        problemas = ["Sin respuesta"]
        if not r["error"] and r["response_text"]:
            try:
                RespuestaRAG.model_validate_json(r["response_text"])
                schema_ok += 1
            except ValidationError:
                pass
            _, problemas = validar_salida(r["response_text"], {x["chunk_id"] for x in registros_base[id]["retrieved"]})
        contract_ok += int(not problemas and not r["error"])
        status_ok += int(r["parsed"] is not None and r["parsed"]["status"] == casos[id]["expected_status"])
        errores += int(bool(r["error"]))
    fila = {"variant": variante, "n": len(casos), "schema_ok": schema_ok,
            "local_contract_ok": contract_ok, "expected_status_ok": status_ok, "technical_errors": errores}
    resumen.append(fila)
    print(fila)

resumen_experimento = {"retrieval": comparacion_retrieval, "filters": experimentos_filtro, "generation": resumen}
(corrida / "metrics.json").write_text(json.dumps(resumen_experimento, ensure_ascii=False, indent=2), encoding="utf-8")
ruta_revision = corrida / "semantic_review.json"
if not ruta_revision.exists():
    fichas = {"reviewer": "pendiente", "cases": []}
    for r in resultados:
        e = casos[r["case_id"]]
        fichas["cases"].append({"variant": r["variant"], "case_id": r["case_id"],
            "required_facts": [{"criterion": f, "met": None, "evidence": ""} for f in e["required_facts"]],
            "must_not_use": [{"criterion": f, "violated": None, "evidence": ""} for f in e["must_not_use"]],
            "grounded": None, "diagnosis": "pendiente"})
    ruta_revision.write_text(json.dumps(fichas, ensure_ascii=False, indent=2), encoding="utf-8")
print("Métricas y fichas:", corrida.relative_to(raiz))

## 10. Inspeccionar un caso y decidir qué adoptar

Mirar juntos el prompt recibido, las tres respuestas y la rúbrica permite encontrar diferencias que los contadores ocultan. Se muestra Q08 por defecto; cambiar `INSPECCIONAR` permite revisar otro caso sin llamadas nuevas.

Una diferencia observada no garantiza que la variante mejorada nunca repita el error. Para una evaluación más seria necesitaríamos repeticiones, casos reservados y anotaciones revisadas por el docente. Tampoco medimos aquí el efecto combinado de híbrido + filtros + prompt: aplicarlos todos y atribuir el resultado a uno solo sería incorrecto.

In [ ]:
INSPECCIONAR = "Q08"
print("Pregunta:", preguntas[INSPECCIONAR])
print("BASE:", registros_base[INSPECCIONAR]["response_text"])
for variante in VARIANTES:
    r = next(r for r in resultados if r["variant"] == variante and r["case_id"] == INSPECCIONAR)
    print("\n", variante, r.get("response_text"), "error:", r["error"])
print("\nRÚBRICA:", json.dumps(casos[INSPECCIONAR], ensure_ascii=False, indent=2))
print("\nEVIDENCIA IDÉNTICA EN LAS TRES CONDICIONES:\n", registros_base[INSPECCIONAR]["input"])

## Cierre del ejercicio

Ya recorrimos preparación, recuperación, generación, evaluación, organización con Haystack y mejoras controladas. Cada capa puede fallar de una manera distinta. La selección de un buen modelo o framework no elimina la necesidad de inspeccionar fuentes y medir resultados.

Antes de adoptar cambios: revisar si la búsqueda mejora las secciones necesarias, si el filtro respeta la intención histórica, si el esquema elimina fallos de forma y si las nuevas instrucciones reducen inferencias injustificadas sin aumentar abstenciones innecesarias. Documentar también regresiones.

Los resultados de este notebook son material didáctico de desarrollo. No se modificaron las preguntas para favorecer una variante ni se cambiaron las soluciones esperadas. Las mejoras propuestas no garantizan resolver los diez casos.

Fuentes: [BM25 en Haystack](https://docs.haystack.deepset.ai/docs/inmemorybm25retriever), [fusión por rankings](https://docs.haystack.deepset.ai/docs/documentjoiner), [salidas estructuradas de OpenAI](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=responses).

## Cierre: comparar tus resultados

Compara las tres filas de métricas de tu ejecución: base, solo esquema y esquema con instrucciones. Completa las fichas de `semantic_review.json` con evidencia concreta. ¿Mejoró el formato? ¿Mejoró la abstención? ¿Las citas respaldan realmente las respuestas? Inspecciona especialmente Q02, Q08 y Q10 sin presuponer qué responderá el modelo.

Compara también dense, BM25 y RRF: una mejor cobertura documental no garantiza que la sección recuperada contenga la regla necesaria. Explica cuándo el filtro de vigencia ayuda y cuándo pierde contexto histórico. Escribe tus conclusiones y conserva los resultados localmente. No se garantiza que una variante gane; las respuestas pueden variar entre ejecuciones.